In [207]:
from pathlib import Path
import json
import re


def load_benchmark_files(
    directory="benchmarks",
    throughput=None,
    n_scans_per_batch=None,
    n_workers=None,
    n_partitions=None,
    verify_content=True,
):
    """
    Load benchmark JSON files matching requested parameters.

    Parameters
    ----------
    directory : str
        Benchmark directory.
    throughput : int | None
        Required throughput in MBps. None means any.
    n_scans_per_batch : int | None
        Required scans per batch. None means any.
    n_partitions : int | None
        Required partition count. None means any.
    verify_content : bool
        If True, verify filename metadata matches JSON content.

    Returns
    -------
    list[dict]
        List of dicts with:
        {
            "path": Path,
            "throughput": int,
            "spb": int,
            "parts": int,
            "data": dict
        }
    """

    # Build glob pattern
    throughput_pattern = f"{throughput}MBps" if throughput is not None else "*MBps"
    
    nspb_pattern = f"{n_scans_per_batch}SpB" if n_scans_per_batch is not None else "*SpB"
    nparts_pattern = f"{n_partitions}parts" if n_partitions is not None else "*parts"
    nws_pattern = f"{n_workers}ws" if n_workers is not None else "*ws"

    pattern = f"*--{throughput_pattern}--{nspb_pattern}--{nws_pattern}--{nparts_pattern}.json"

    filename_re = re.compile(
        r".+--(?P<throughput>\d+)MBps--"
        r"(?P<spb>\d+)SpB--"
        r"(?P<ws>\d+)ws--"
        r"(?P<parts>\d+)parts\.json$"
    )

    results = []
    skipped_incomplete = []

    for path in Path(directory).glob(pattern):
        match = filename_re.match(path.name)
        if not match:
            continue

        filename_values = {
            "throughput_MB": int(match.group("throughput")),
            "n_scans_per_batch": int(match.group("spb")),
            "n_workers": int(match.group("ws")),
            "n_partitions": int(match.group("parts")),
        }

        with path.open() as f:
            content = json.load(f)

            # analysis_end_ts can be null if the dashboard/analysis process
            # was killed or crashed before finishing this run (the producer
            # side still wrote the file). Every metric needs
            # analysis_end_ts - producer_begin_ts, so these runs are
            # unusable - skip them here instead of failing downstream.
            if content.get("analysis_end_ts") is None:
                skipped_incomplete.append(path)
                continue

            # Adjust these keys if your JSON uses different names
            content_values = {
                "throughput_MB": content["throughput_MB"],
                "n_scans_per_batch": content["n_scans_per_batch"],
                "n_workers": content["n_workers"],
                "n_partitions": content["n_partitions"],
            }

            #if filename_values != content_values:
            #    raise ValueError(
            #        f"Filename/content mismatch:\n"
            #        f"File: {path}\n"
            #        f"Filename values: {filename_values}\n"
            #        f"JSON values: {content_values}"
            #    )

            results.append(
                {
                    "path": path,
                    **filename_values,
                    "data": content,
                })

    if skipped_incomplete:
        print(
            f"Skipped {len(skipped_incomplete)} incomplete run(s) with analysis_end_ts=null: "
            + ", ".join(p.name for p in skipped_incomplete)
        )

    return results


In [208]:
benchmarks = load_benchmark_files(
    directory="benchmarks",
    throughput=16,
    n_partitions=3
)

for b in benchmarks:
    print(
        b["path"],
        b["n_scans_per_batch"],
        b["n_partitions"],
        b["n_workers"]
    )

benchmarks/2026-07-15_17-35-04--16MBps--512SpB--3ws--3parts.json 512 3 3
benchmarks/2026-07-15_18-26-42--16MBps--2048SpB--3ws--3parts.json 2048 3 3
benchmarks/2026-07-15_17-58-41--16MBps--1024SpB--3ws--3parts.json 1024 3 3
benchmarks/2026-07-15_17-40-53--16MBps--1024SpB--3ws--3parts.json 1024 3 3
benchmarks/2026-07-15_18-29-37--16MBps--4096SpB--3ws--3parts.json 4096 3 3
benchmarks/2026-07-15_18-50-20--16MBps--4096SpB--3ws--3parts.json 4096 3 3
benchmarks/2026-07-15_18-46-26--16MBps--4096SpB--3ws--3parts.json 4096 3 3
benchmarks/2026-07-15_17-32-10--16MBps--512SpB--3ws--3parts.json 512 3 3
benchmarks/2026-07-15_18-07-24--16MBps--2048SpB--3ws--3parts.json 2048 3 3
benchmarks/2026-07-15_17-55-47--16MBps--1024SpB--3ws--3parts.json 1024 3 3
benchmarks/2026-07-15_18-19-43--16MBps--2048SpB--3ws--3parts.json 2048 3 3


# Benchmark metrics vs n_workers / partition_worker_ratio (generalized)

In [ ]:
from pathlib import Path
from collections import defaultdict
import json as _json
import math

import matplotlib.pyplot as plt
import numpy as np


# From the Producer configuration (see producer.ipynb): each scan has
# N_SAMPLES_PER_SCAN=2048 samples, and each sample is an (I, Q) pair of
# float32 values -> 8 bytes/sample -> 16384 bytes/scan.
BYTES_PER_SCAN = 2048 * 8


def _dedup_records(records):
    """Defensive: keep only exact-duplicate protection in case some files
    still have it (older format runs had this - the current format doesn't,
    but this is a harmless no-op when there's nothing to remove)."""
    seen = set()
    unique = []
    for r in records:
        key = _json.dumps(r, sort_keys=True)
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique


def _drop_stray_records(b):
    """Defensive: drop any record whose receive_ts falls outside this run's
    own [producer_begin_ts, analysis_end_ts] window (older format runs could
    carry over records from a previous run - the current format doesn't, but
    this is a harmless no-op when there's nothing to remove)."""
    begin = b["data"]["producer_begin_ts"]
    end = b["data"]["analysis_end_ts"]
    return [r for r in b["data"]["records"] if begin <= r["receive_ts"] <= end]


def _records(b):
    return _dedup_records(_drop_stray_records(b))


def _record_values(b, field):
    """One value per record for a scalar per-batch field (e.g.
    processing_time_ms, batch_latency_ms) - current format logs one batch
    per record, so no flattening of nested lists is needed anymore."""
    return [record[field] for record in _records(b)]


class _NoDataError(Exception):
    """Raised by a metric's extract() when a run has no usable records for
    it (empty "records", or everything got dropped by dedup/stray-filtering)
    - signals the caller to skip that run instead of silently averaging in
    NaN (np.mean([]) warns and returns NaN rather than raising) or crashing
    downstream (np.percentile([]) raises IndexError)."""


def _mean_record_field(b, field):
    values = _record_values(b, field)
    if not values:
        raise _NoDataError(f"no usable {field!r} records in {b['path']}")
    return np.mean(values)


def _achieved_throughput_MBps(b):
    """Effective end-to-end throughput: data volume actually delivered to the
    consumer during THIS run, divided by the elapsed time. The current
    format reports n_total_scans directly, so no reconstruction from
    per-record batch counts is needed."""
    total_MB = b["data"]["n_total_scans"] * BYTES_PER_SCAN / (1024 ** 2)
    elapsed = b["data"]["analysis_end_ts"] - b["data"]["producer_begin_ts"]
    return total_MB / elapsed


# Registry of measurable quantities. Each metric only needs to describe how to
# extract a single scalar per benchmark run ("extract"), a y-axis label, and
# a plot title; everything else (grouping, styling, adaptive x-axis, faceted
# grid layout) is shared.
#
# "target"/"ylim" together decide the drawing mode in _draw_metric_panel:
#   - target is not None  -> this metric has a known expected value (a
#     dashed reference line makes sense) -> ALWAYS drawn as a scatter/line
#     plot, with a y-axis zoomed in around that reference line, regardless
#     of what's on the x-axis (n_workers, partition_worker_ratio, n_scans).
#   - target is None -> no known expected value -> ALWAYS drawn as a bar
#     chart, with the y-axis scaled to the data (0 .. max*1.2).
METRICS = {
    "total_time": {
        "title": "Total Time",
        "extract": lambda b: b["data"]["analysis_end_ts"] - b["data"]["producer_begin_ts"],
        "ylabel": "average tot time (s)",
        "target": lambda throughput: 1920 / throughput,
        "target_label": "target time",
        # Zoomed around the target time instead of the old (0, 2x) range.
        "ylim": lambda throughput: (0.5 * (1920 / throughput), 2.2 * (1920 / throughput)),
    },
    "achieved_throughput_MBps": {
        "title": "Achieved Throughput (MB/s)",
        "extract": _achieved_throughput_MBps,
        "ylabel": "achieved throughput (MB/s)",
        "target": lambda throughput: throughput,
        "target_label": "target throughput",
        "ylim": lambda throughput: (10, 17.5),
    },
    "avg_processing_time_ms": {
        "title": "Avg Processing Time (ms)",
        "extract": lambda b: _mean_record_field(b, "processing_time_ms"),
        "ylabel": "average processing time (ms)",
        "target": None,
        "ylim": None,
    },
    "avg_net_latency_ms": {
        "title": "Avg Network Latency (ms)",
        "extract": lambda b: _mean_record_field(b, "batch_latency_ms"),
        "ylabel": "average network latency (ms)",
        "target": None,
        "ylim": None,
    },
}

_SHORT_NAMES = {"throughput": "MBps", "n_scans": "SpB", "partition_worker_ratio": "pwr", "n_workers": "ws"}
_BAR_COLOR = "#3B78A6"
_ACCENT_COLOR = "#22303F"


def _compute_points(benchmarks, get_x, extract_value):
    """Group benchmark runs by x-value and reduce each group to
    {"x", "mean", "std", "files"} - shared by every plotting function.
    Runs where extract_value() raises _NoDataError (e.g. no usable records
    for this metric) are skipped, with a warning listing which files."""
    grouped = defaultdict(list)
    skipped = []
    for b in benchmarks:
        x = get_x(b)
        try:
            value = extract_value(b)
        except _NoDataError:
            skipped.append(b["path"])
            continue
        grouped[x].append({"value": value, "file": b["path"]})

    if skipped:
        print(f"Skipped {len(skipped)} run(s) with no usable data for this metric: "
              + ", ".join(p.name for p in skipped))

    points = []
    for x, runs in grouped.items():
        values = [r["value"] for r in runs]
        points.append({
            "x": x, "mean": np.mean(values), "std": np.std(values),
            "files": [r["file"] for r in runs],
        })
    points.sort(key=lambda p: p["x"])
    return points


def _draw_metric_panel(ax, points, variable, metric_spec, panel_fixed_dict):
    """Draw one metric-vs-variable panel into a given Axes. Two modes,
    chosen by whether the metric has a known target value:
      - has a target -> scatter/line, y-axis zoomed around the target line,
        regardless of what `variable` is (n_workers, partition_worker_ratio,
        or n_scans).
      - no target -> bar chart, y-axis scaled to the data.
    Used both standalone and as one subplot of a faceted grid - the shared
    "<metric> vs <variable>" title lives once on the figure (see callers),
    not repeated per panel."""
    xs = [p["x"] for p in points]
    means = [p["mean"] for p in points]
    errors = [p["std"] for p in points]
    n_points = len(points)

    scatter_mode = metric_spec["target"] is not None
    throughput = panel_fixed_dict.get("throughput")

    if scatter_mode:
        ax.errorbar(
            xs, means, yerr=errors, fmt="o-", color=_BAR_COLOR, ecolor=_ACCENT_COLOR,
            linewidth=1.8, markersize=6.5, markeredgecolor="white", capsize=4, zorder=3,
        )
        if variable == "n_scans":
            ax.set_xscale("log", base=2)
            if n_points > 1:
                pad = (max(xs) / min(xs)) ** 0.2
                ax.set_xlim(min(xs) / pad, max(xs) * pad)
        else:
            ax.set_xlim(min(xs) - 0.6, max(xs) + 0.6)
        ax.set_xticks(xs)
        ax.set_xticklabels([str(x) for x in xs])
        ax.minorticks_off()

        for p in points:
            n = len(p["files"])
            sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
            ax.annotate(
                f"{p['mean']:.2f} \u00b1 {sem:.2f} (n={n})",
                (p["x"], p["mean"]), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=7.5, color="#333333",
            )
    else:
        positions = np.arange(n_points)
        bars = ax.bar(
            positions, means, yerr=errors, capsize=4, width=0.6,
            color=_BAR_COLOR, edgecolor="white", linewidth=1.2,
            error_kw=dict(ecolor=_ACCENT_COLOR, elinewidth=1.2, capthick=1.2),
            zorder=3,
        )
        ax.set_xticks(positions)
        ax.set_xticklabels([str(x) for x in xs])
        ax.set_xlim(-0.7, n_points - 0.3)
        for bar, p in zip(bars, points):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, str(len(p["files"])),
                ha="center", va="center", fontsize=10, fontweight="bold", color="white",
            )

    ax.set_xlabel(variable, fontsize=10.5)
    ax.set_ylabel(metric_spec["ylabel"], fontsize=10.5)
    ax.set_title(
        " ".join(f"{k}={v}" for k, v in panel_fixed_dict.items()),
        fontsize=10.5, color="#666666", pad=10,
    )

    legend_handles, legend_labels = [], []
    if metric_spec["ylim"] is not None:
        if throughput is None:
            raise ValueError("throughput must be fixed to use this metric's y-scale")
        ax.set_ylim(*metric_spec["ylim"](throughput))
        if metric_spec["target"] is not None:
            target = metric_spec["target"](throughput)
            target_line = ax.axhline(target, linestyle="--", color=_ACCENT_COLOR, linewidth=1.3, zorder=2)
            legend_handles.append(target_line)
            legend_labels.append(metric_spec["target_label"])
    else:
        y_max = max(m + e for m, e in zip(means, errors)) * 1.2
        ax.set_ylim(0, y_max if y_max > 0 else 1)

    if not scatter_mode:
        # Bar mode: legend lists mean +/- SEM for each bar too.
        for bar, p in zip(ax.patches, points):
            n = len(p["files"])
            sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
            legend_handles.append(bar)
            legend_labels.append(f"{variable}={p['x']}: {p['mean']:.2f} \u00b1 {sem:.2f}")

    if legend_handles:
        ax.legend(legend_handles, legend_labels, loc="best", fontsize=7.5 if not scatter_mode else 8.5, frameon=False)

    ax.grid(axis="y" if not scatter_mode else "both", alpha=0.25, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)


def plot_metric_vs(variable, metric="total_time", facet_variable="n_scans",
                    benchmark_dir="benchmarks", output_dir="results"):
    """Plot `metric` vs `variable`, one grid image per remaining fixed
    combination, faceting into subplots over `facet_variable` (e.g. one grid
    per throughput/partition_worker_ratio combo, with one subplot per
    n_scans value) - instead of a separate image per n_scans value."""
    assert variable in {"throughput", "n_scans", "partition_worker_ratio", "n_workers"}
    assert facet_variable in {"throughput", "n_scans", "partition_worker_ratio", "n_workers"}
    assert facet_variable != variable
    assert metric in METRICS, f"Unknown metric {metric!r}, choose from {sorted(METRICS)}"
    metric_spec = METRICS[metric]

    output_dir = Path(output_dir) / metric
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(directory=benchmark_dir, verify_content=True)

    def partition_worker_ratio(b):
        n_parts, n_workers = b["n_partitions"], b["n_workers"]
        assert n_parts % n_workers == 0, (
            f"n_partitions ({n_parts}) is not a multiple of n_workers ({n_workers}) "
            f"for {b['path']}"
        )
        return n_parts // n_workers

    get_value = {
        "throughput": lambda b: b["throughput_MB"],
        "n_scans": lambda b: b["data"]["n_scans_per_batch"],
        "partition_worker_ratio": partition_worker_ratio,
        "n_workers": lambda b: b["n_workers"],
    }

    fixed_variables = [v for v in get_value if v not in (variable, facet_variable)]

    # panels[outer_fixed][facet_value] = list of {"x", "mean", "std", "files"}
    grouped = defaultdict(list)
    skipped = []
    for b in benchmarks:
        x = get_value[variable](b)
        facet = get_value[facet_variable](b)
        outer_fixed = tuple(get_value[v](b) for v in fixed_variables)
        try:
            value = metric_spec["extract"](b)
        except _NoDataError:
            skipped.append(b["path"])
            continue
        grouped[(outer_fixed, facet, x)].append({"value": value, "file": b["path"]})

    if skipped:
        print(f"Skipped {len(skipped)} run(s) with no usable data for '{metric}': "
              + ", ".join(p.name for p in skipped))

    panels = defaultdict(lambda: defaultdict(list))
    for (outer_fixed, facet, x), runs in grouped.items():
        values = [r["value"] for r in runs]
        panels[outer_fixed][facet].append({
            "x": x,
            "mean": np.mean(values),
            "std": np.std(values),
            "files": [r["file"] for r in runs],
        })

    for outer_fixed, facets_dict in panels.items():
        outer_fixed_dict = dict(zip(fixed_variables, outer_fixed))
        facet_values = sorted(facets_dict)
        n_facets = len(facet_values)

        ncols = 2 if n_facets > 1 else 1
        nrows = math.ceil(n_facets / ncols)

        fig_w = max(4.5, 1.6 * max(len(v) for v in facets_dict.values()) + 2.2)
        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w * ncols, 5.6 * nrows), squeeze=False)
        axes_flat = axes.flatten()

        for ax, facet_value in zip(axes_flat, facet_values):
            panel_fixed_dict = dict(outer_fixed_dict)
            panel_fixed_dict[facet_variable] = facet_value
            # Keep display order: throughput, n_scans, partition_worker_ratio, n_workers
            panel_fixed_dict = {k: panel_fixed_dict[k] for k in get_value if k in panel_fixed_dict}
            points = sorted(facets_dict[facet_value], key=lambda p: p["x"])
            _draw_metric_panel(ax, points, variable, metric_spec, panel_fixed_dict)

        # Hide any unused subplot slots (e.g. 3 facets in a 2x2 grid).
        for ax in axes_flat[n_facets:]:
            ax.set_visible(False)

        # Single shared title for the whole grid (was previously repeated on
        # every subplot, which collided with the neighboring subplot's title
        # in a multi-column layout).
        fig.suptitle(f"{metric_spec['title']} vs {variable}", fontsize=16, fontweight="bold")

        fig.tight_layout()
        fig.subplots_adjust(top=1 - 0.77 / (5.6 * nrows), wspace=0.3, hspace=0.5)

        short_names = _SHORT_NAMES
        fixed_string = "__".join(f"{k}{v}{short_names[k]}" for k, v in outer_fixed_dict.items())
        filename = f"{metric}_vs_{variable}__{fixed_string}.png"

        fig.savefig(output_dir / filename, dpi=300)
        plt.close(fig)

        txt_filename = filename.replace(".png", ".txt")
        with (output_dir / txt_filename).open("w") as f:
            f.write(f"metric={metric}\n")
            f.write(f"variable={variable}\n")
            f.write(f"facet_variable={facet_variable}\n")
            for k, v in outer_fixed_dict.items():
                f.write(f"{k}={v}\n")
            f.write("\n")

            for facet_value in facet_values:
                f.write(f"### {facet_variable}={facet_value}\n")
                for p in sorted(facets_dict[facet_value], key=lambda p: p["x"]):
                    n = len(p["files"])
                    sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
                    f.write(f"{variable}={p['x']}\n")
                    f.write(f"mean_{metric}={p['mean']}\n")
                    f.write(f"std_{metric}={p['std']}\n")
                    f.write(f"sem_{metric}={sem}\n")
                    for file in p["files"]:
                        f.write(f"    {file}\n")
                    f.write("\n")

In [210]:
for metric in ["total_time", "achieved_throughput_MBps", "avg_processing_time_ms", "avg_net_latency_ms"]:
    plot_metric_vs("n_workers", metric)
    plot_metric_vs("partition_worker_ratio", metric)

# Latency distributions (network/Kafka vs Dask FFT computation)

In [211]:
import matplotlib.pyplot as plt
import numpy as np


def _draw_latency_hist(ax, values, title, color, bins):
    values = np.asarray(values)
    mean = values.mean()
    p50 = np.percentile(values, 50)
    p95 = np.percentile(values, 95)

    ax.hist(values, bins=bins, color=color, edgecolor="black", alpha=0.75)

    mean_line = ax.axvline(mean, color="red", linestyle="--", linewidth=1.3)
    p50_line = ax.axvline(p50, color="orange", linestyle="--", linewidth=1.3)
    p95_line = ax.axvline(p95, color="green", linestyle="--", linewidth=1.3)

    ax.legend(
        [mean_line, p50_line, p95_line],
        [f"Mean: {mean:.2f} ms", f"P50: {p50:.2f} ms", f"P95: {p95:.2f} ms"],
        loc="upper right", fontsize=9,
    )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Frequency (Number of Packets)")
    ax.grid(True, alpha=0.3)

    return {"mean": float(mean), "p50": float(p50), "p95": float(p95), "n": len(values)}


def plot_latency_histograms(
    benchmark_dir="benchmarks", output_dir="results/latencies",
    throughput=None, n_scans_per_batch=None, n_workers=None, n_partitions=None,
    bins=30,
):
    """One latency-histogram figure PER benchmark run (not pooled across
    runs): network+Kafka delivery latency (batch_latency_ms) side by side
    with the Dask worker's own FFT computation time (processing_time_ms),
    both from that single run's records. Filters work the same as
    load_benchmark_files (None = any) and just narrow down which runs get a
    figure."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(
        directory=benchmark_dir, throughput=throughput, n_scans_per_batch=n_scans_per_batch,
        n_workers=n_workers, n_partitions=n_partitions, verify_content=True,
    )

    skipped = []
    for b in benchmarks:
        net_latencies = _record_values(b, "batch_latency_ms")
        processing_times = _record_values(b, "processing_time_ms")

        if not net_latencies or not processing_times:
            # No usable records for this run (empty "records", or everything
            # got dropped by dedup/stray-filtering) - np.percentile([]) would
            # raise IndexError, so skip instead of crashing.
            skipped.append(b["path"])
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

        net_stats = _draw_latency_hist(axes[0], net_latencies, "Network & Kafka Pipeline Latency", "#6D8FC7", bins)
        proc_stats = _draw_latency_hist(axes[1], processing_times, "Dask Worker FFT Computation Latency", "#7DB893", bins)

        run_tag = (
            f"{b['throughput_MB']}MBps__{b['n_scans_per_batch']}SpB__"
            f"{b['n_workers']}ws__{b['n_partitions']}parts__{b['path'].stem}"
        )
        fig.suptitle(run_tag, fontsize=10, color="#666666", y=1.02)
        fig.tight_layout()

        filename = f"latency_histograms__{run_tag}.png"
        fig.savefig(output_dir / filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

        with (output_dir / filename.replace(".png", ".txt")).open("w") as f:
            f.write(f"run={b['path']}\n")
            f.write(f"throughput={b['throughput_MB']}\n")
            f.write(f"n_scans={b['n_scans_per_batch']}\n")
            f.write(f"n_workers={b['n_workers']}\n")
            f.write(f"n_partitions={b['n_partitions']}\n")
            f.write("\n")
            f.write("batch_latency_ms: " + ", ".join(f"{k}={v}" for k, v in net_stats.items()) + "\n")
            f.write("processing_time_ms: " + ", ".join(f"{k}={v}" for k, v in proc_stats.items()) + "\n")

    if skipped:
        print(f"Skipped {len(skipped)} run(s) with no usable latency records: "
              + ", ".join(p.name for p in skipped))


In [212]:
plot_latency_histograms()

# Metric trend vs n_scans_per_batch (n_workers/n_partitions fixed)

In [213]:
from pathlib import Path
from collections import defaultdict
import math

import matplotlib.pyplot as plt
import numpy as np


def _partition_worker_ratio(b):
    n_parts, n_workers = b["n_partitions"], b["n_workers"]
    assert n_parts % n_workers == 0, (
        f"n_partitions ({n_parts}) is not a multiple of n_workers ({n_workers}) for {b['path']}"
    )
    return n_parts // n_workers


def plot_metric_scans_grid(metric, throughput=None, benchmark_dir="benchmarks", output_dir="results"):
    """Metric vs n_scans_per_batch, one grid image covering EVERY
    (n_workers, n_partitions) combination present in the data - nothing
    aggregated together. Rows = n_workers (sorted), columns =
    partition_worker_ratio (sorted), matching the layout used for the
    n_workers/ratio plots. A cell is left empty if that combination isn't
    present in the data. Draw mode (scatter with zoomed y-axis vs. bar
    chart) is decided by the metric, same as plot_metric_vs."""
    assert metric in METRICS, f"Unknown metric {metric!r}, choose from {sorted(METRICS)}"
    metric_spec = METRICS[metric]

    output_dir = Path(output_dir) / metric
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(directory=benchmark_dir, throughput=throughput, verify_content=True)
    if not benchmarks:
        raise ValueError(f"No benchmark files found (throughput={throughput})")

    by_cell = defaultdict(list)
    for b in benchmarks:
        by_cell[(b["n_workers"], _partition_worker_ratio(b))].append(b)

    workers_values = sorted({ws for ws, _ in by_cell})
    ratio_values = sorted({r for _, r in by_cell})
    nrows, ncols = len(workers_values), len(ratio_values)

    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2 * ncols, 5.6 * nrows), squeeze=False)

    txt_sections = []
    for i, ws in enumerate(workers_values):
        for j, ratio in enumerate(ratio_values):
            ax = axes[i][j]
            cell_benchmarks = by_cell.get((ws, ratio))
            if not cell_benchmarks:
                ax.set_visible(False)
                continue

            n_partitions = ws * ratio
            throughput_values = {b["throughput_MB"] for b in cell_benchmarks}
            panel_fixed_dict = {"n_workers": ws, "n_partitions": n_partitions}
            if len(throughput_values) == 1:
                panel_fixed_dict = {"throughput": throughput_values.pop(), **panel_fixed_dict}

            points = _compute_points(cell_benchmarks, lambda b: b["n_scans_per_batch"], metric_spec["extract"])
            _draw_metric_panel(ax, points, "n_scans", metric_spec, panel_fixed_dict)
            txt_sections.append((panel_fixed_dict, points))

    fig.suptitle(f"{metric_spec['title']} vs n_scans_per_batch", fontsize=16, fontweight="bold")
    fig.tight_layout()
    fig.subplots_adjust(top=1 - 0.77 / (5.6 * nrows), wspace=0.32, hspace=0.55)

    throughput_tag = f"__{throughput}MBps" if throughput is not None else ""
    filename = f"{metric}_vs_n_scans__grid{throughput_tag}.png"
    fig.savefig(output_dir / filename, dpi=300)
    plt.close(fig)

    txt_filename = filename.replace(".png", ".txt")
    with (output_dir / txt_filename).open("w") as f:
        f.write(f"metric={metric}\n\n")
        for panel_fixed_dict, points in txt_sections:
            f.write("### " + " ".join(f"{k}={v}" for k, v in panel_fixed_dict.items()) + "\n")
            for p in points:
                n = len(p["files"])
                sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
                f.write(f"n_scans={p['x']}\n")
                f.write(f"mean_{metric}={p['mean']}\n")
                f.write(f"std_{metric}={p['std']}\n")
                f.write(f"sem_{metric}={sem}\n")
                for file in p["files"]:
                    f.write(f"    {file}\n")
                f.write("\n")


def plot_metric_vs_scans(metric, n_workers, n_partitions, throughput=None,
                          benchmark_dir="benchmarks", output_dir="results"):
    """Single-panel version of the same trend/bar plot, for one specific
    (n_workers, n_partitions) combination. Prefer plot_metric_scans_grid to
    see every combination at once without aggregating them together."""
    assert metric in METRICS, f"Unknown metric {metric!r}, choose from {sorted(METRICS)}"
    metric_spec = METRICS[metric]

    output_dir = Path(output_dir) / metric
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(
        directory=benchmark_dir, throughput=throughput,
        n_workers=n_workers, n_partitions=n_partitions, verify_content=True,
    )
    if not benchmarks:
        raise ValueError(
            f"No benchmark files match n_workers={n_workers}, n_partitions={n_partitions}, "
            f"throughput={throughput}"
        )

    fixed_dict = {}
    for key, attr in [("throughput", "throughput_MB"), ("n_workers", "n_workers"), ("n_partitions", "n_partitions")]:
        values_seen = {b[attr] for b in benchmarks}
        if len(values_seen) == 1:
            fixed_dict[key] = values_seen.pop()

    points = _compute_points(benchmarks, lambda b: b["n_scans_per_batch"], metric_spec["extract"])

    fig, ax = plt.subplots(figsize=(7.5, 5.8))
    _draw_metric_panel(ax, points, "n_scans", metric_spec, fixed_dict)
    ax.set_title(f"{metric_spec['title']} vs n_scans_per_batch", fontsize=14, fontweight="bold", pad=10)
    fig.tight_layout()

    tag_parts = []
    if "n_workers" in fixed_dict:
        tag_parts.append(f"{fixed_dict['n_workers']}ws")
    if "n_partitions" in fixed_dict:
        tag_parts.append(f"{fixed_dict['n_partitions']}parts")
    if "throughput" in fixed_dict:
        tag_parts.append(f"{fixed_dict['throughput']}MBps")
    tag = "__".join(tag_parts) if tag_parts else "all_runs"
    filename = f"{metric}_vs_n_scans__{tag}.png"

    fig.savefig(output_dir / filename, dpi=300)
    plt.close(fig)

    txt_filename = filename.replace(".png", ".txt")
    with (output_dir / txt_filename).open("w") as f:
        f.write(f"metric={metric}\n")
        for k, v in fixed_dict.items():
            f.write(f"{k}={v}\n")
        f.write("\n")
        for p in points:
            n = len(p["files"])
            sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
            f.write(f"n_scans={p['x']}\n")
            f.write(f"mean_{metric}={p['mean']}\n")
            f.write(f"std_{metric}={p['std']}\n")
            f.write(f"sem_{metric}={sem}\n")
            for file in p["files"]:
                f.write(f"    {file}\n")
            f.write("\n")

In [214]:
for metric in ["achieved_throughput_MBps", "avg_net_latency_ms", "avg_processing_time_ms", "total_time"]:
    plot_metric_scans_grid(metric)